In [2]:
# ---------------------------------------------------------
# AIM:
# To perform object detection using YOLOv8 and display output
# directly using OpenCV
# ---------------------------------------------------------

# ---------------------------------------------------------
# Import Libraries
# ---------------------------------------------------------
import cv2
import random
from ultralytics import YOLO

# ---------------------------------------------------------
# Load YOLO Model
# ---------------------------------------------------------
model = YOLO("yolov8n.pt")

# ---------------------------------------------------------
# 1. OBJECT DETECTION ON DEFAULT IMAGE
# ---------------------------------------------------------
print("\nRunning detection on default image...")

# YOLO will automatically download this image
results = model("https://ultralytics.com/images/bus.jpg")

# Get output image
output_img = results[0].plot()

# Convert RGB → BGR for OpenCV
output_img = cv2.cvtColor(output_img, cv2.COLOR_RGB2BGR)

# Show output
cv2.imshow("YOLO Detection - Image", output_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

# ---------------------------------------------------------
# 2. OBJECT DETECTION ON VIDEO
# ---------------------------------------------------------
print("\nRunning detection on video...")

video_path = "input.mp4"   # Place your video in same folder

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Video not found")
    exit()

fps = int(cap.get(cv2.CAP_PROP_FPS))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter("output.mp4",
                      cv2.VideoWriter_fourcc(*'mp4v'),
                      fps, (w, h))

# Color generator
def get_color(cls_id):
    random.seed(cls_id)
    return tuple(random.randint(0, 255) for _ in range(3))

frame_count = 0
MAX_FRAMES = 100

while True:
    ret, frame = cap.read()
    if not ret or frame_count >= MAX_FRAMES:
        break

    results = model.track(frame, stream=True, verbose=False)

    for result in results:
        for box in result.boxes:

            if box.conf[0] > 0.4:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cls = int(box.cls[0])
                conf = float(box.conf[0])

                color = get_color(cls)

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

                label = f"{result.names[cls]} {conf:.2f}"
                cv2.putText(frame, label, (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.6, color, 2)

    # Show live output
    cv2.imshow("YOLO Detection - Video", frame)

    out.write(frame)
    frame_count += 1

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()

print("\n✅ Output video saved as output.mp4")


Running detection on default image...

Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
image 1/1 C:\Users\karth\bus.jpg: 640x480 4 persons, 1 bus, 1 stop sign, 45.9ms
Speed: 1.5ms preprocess, 45.9ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 480)

Running detection on video...
Error: Video not found

✅ Output video saved as output.mp4
